In [1]:
pip install bert-score nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 1: Configuration
import os
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import Dataset
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score
from bert_score import score
import nltk
from tqdm import tqdm
# nltk.download('wordnet')

2025-07-21 10:02:46.612886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753092166.635011     773 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753092166.641785     773 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Configuration dictionary for easy customization
CONFIG = {
    'model_name': 't5-large',
    'max_input_length': 512,
    'max_target_length': 128,
    'batch_size': 1,
    'epochs': 10,
    'learning_rate': 2e-5,
    'test_size': 0.2,
    'random_state': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'output_dir': './t5_model',
    'model_save_path': './t5_model/t5_requirements_to_userstory'
}

In [4]:
# Load and prepare dataset
def load_and_prepare_data(file_path):
    # Load dataset
    df = pd.read_csv(file_path)
    
    # Ensure 'Functional Requirement' and 'User Story' columns are strings, handling NaN/float
    df['Functional Requirement'] = df['Functional Requirement'].fillna('').astype(str)
    df['User Story'] = df['User Story'].fillna('').astype(str)
    
    # Group functional requirements by User Story ID to create input-output pairs
    grouped = df.groupby('User Story ID').agg({
        'User Story': 'first',
        'Functional Requirement': lambda x: ' '.join(x)
    }).reset_index()
    
    # Create dataset
    data = {
        'input_text': ['functional requirement: ' + req for req in grouped['Functional Requirement']],
        'target_text': grouped['User Story']
    }
    dataset = Dataset.from_dict(data)
    
    # Split into train and test
    train_dataset, test_dataset = train_test_split(
        dataset, test_size=CONFIG['test_size'], random_state=CONFIG['random_state']
    )
    
    return Dataset.from_dict(train_dataset), Dataset.from_dict(test_dataset)

In [5]:
# Tokenize dataset for T5 model
def tokenize_dataset(dataset, tokenizer):
    def tokenize_function(examples):
        # Tokenize inputs and targets
        inputs = tokenizer(
            examples['input_text'],
            max_length=CONFIG['max_input_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        targets = tokenizer(
            examples['target_text'],
            max_length=CONFIG['max_target_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs.input_ids,  # Keep as tensor, remove squeeze
            'attention_mask': inputs.attention_mask,
            'labels': targets.input_ids
        }
    
    # Apply tokenization and set format for PyTorch tensors
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized_dataset

In [6]:
# Evaluate model with BLEU, METEOR, and BERTScore
def evaluate_model(model, test_dataset, tokenizer):
    from tqdm import tqdm
    model.eval()
    predictions = []
    references = []
    input_texts = []
    
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset, batch_size=CONFIG['batch_size']
    )
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=CONFIG['max_target_length'] * 3,  # Further increase for complete outputs
                min_length=30,  # Encourage longer, meaningful outputs
                num_beams=8,  # Increase beams for better quality
                length_penalty=0.6,  # Further reduce penalty for longer sequences
                no_repeat_ngram_size=3,  # Prevent repetitive phrases
                early_stopping=True,
                do_sample=True,  # Add sampling for diversity
                top_p=0.9,  # Use nucleus sampling to improve output variety
                temperature=0.7  # Control randomness
            )
            
            preds = [tokenizer.decode(ids, skip_special_tokens=True) for ids in outputs]
            refs = [tokenizer.decode(ids, skip_special_tokens=True) for ids in batch['labels']]
            inputs = [tokenizer.decode(ids, skip_special_tokens=True) for ids in batch['input_ids']]
            
            predictions.extend(preds)
            references.extend(refs)
            input_texts.extend(inputs)
    
    # Debug: Print inputs, predictions, and references
    print("\nDebug: Sample Inputs, Predictions, and References:")
    for i, (inp, pred, ref) in enumerate(zip(input_texts[:5], predictions[:5], references[:5])):
        print(f"Sample {i + 1}:")
        print(f"Input (Functional Requirement): {inp}")
        print(f"Predicted User Story: {pred}")
        print(f"Reference User Story: {ref}")
        print()
    
    # Debug: Print counts of valid and empty predictions
    valid_predictions = [p for p in predictions if p.strip()]
    print(f"Total Predictions: {len(predictions)}")
    print(f"Valid (Non-Empty) Predictions: {len(valid_predictions)}")
    print(f"Empty Predictions: {len(predictions) - len(valid_predictions)}")
    
    # Filter valid prediction-reference pairs
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p.strip() and r.strip()]
    if not valid_pairs:
        print("Warning: No valid prediction-reference pairs for metric calculation.")
        return {
            'bleu': 0.0,
            'meteor': 0.0,
            'bertscore_precision': 0.0,
            'bertscore_recall': 0.0,
            'bertscore_f1': 0.0,
            'predictions': predictions,
            'references': references
        }
    
    valid_predictions, valid_references = zip(*valid_pairs)
    
    # Calculate BLEU and METEOR with smoothing
    bleu_scores = []
    meteor_scores = []
    
    for pred, ref in valid_pairs:
        try:
            bleu_scores.append(sentence_bleu([ref.split()], pred.split(), 
                                          weights=(0.25, 0.25, 0.25, 0.25),
                                          smoothing_function=lambda precisions, **kw: [p + 1e-12 for p in precisions]))
            meteor_scores.append(meteor_score([ref.split()], pred.split()))
        except Exception as e:
            print(f"Error calculating metrics for pred='{pred}', ref='{ref}': {e}")
            continue
    
    # Calculate BERTScore
    bertscore_precision, bertscore_recall, bertscore_f1 = 0.0, 0.0, 0.0
    if valid_predictions and valid_references:
        try:
            P, R, F1 = score(list(valid_predictions), list(valid_references), lang='en', verbose=True)
            bertscore_precision = P.mean().item() if P.numel() > 0 else 0.0
            bertscore_recall = R.mean().item() if R.numel() > 0 else 0.0
            bertscore_f1 = F1.mean().item() if F1.numel() > 0 else 0.0
        except Exception as e:
            print(f"Error calculating BERTScore: {e}")
    
    return {
        'bleu': sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0,
        'meteor': sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0,
        'bertscore_precision': bertscore_precision,
        'bertscore_recall': bertscore_recall,
        'bertscore_f1': bertscore_f1,
        'predictions': predictions,
        'references': references
    }

In [7]:
# Train T5 model with tqdm, table display, multiple GPUs, and early stopping
def train_model(train_dataset, test_dataset, tokenizer):
    import pandas as pd
    from tqdm import tqdm
    import torch
    from torch.nn.parallel import DataParallel
    
    model = T5ForConditionalGeneration.from_pretrained(CONFIG['model_name']).to(CONFIG['device'])
    
    # Wrap model with DataParallel for multiple GPUs if available
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = DataParallel(model)
    
    model = model.to(CONFIG['device'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
    
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=CONFIG['batch_size'], shuffle=True
    )
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset, batch_size=CONFIG['batch_size']
    )
    
    # Early stopping parameters
    patience = 3  # Number of epochs to wait for improvement
    best_val_loss = float('inf')
    epochs_no_improve = 0
    early_stop = False
    
    # Lists to store losses for table
    epoch_data = []
    
    for epoch in range(CONFIG['epochs']):
        if early_stop:
            print(f"Early stopping triggered after {epoch} epochs.")
            break
            
        model.train()
        total_train_loss = 0
        train_loop = tqdm(train_dataloader, desc=f"Training Epoch {epoch + 1}")
        
        for batch in train_loop:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            # Handle DataParallel loss (mean across GPUs)
            loss = outputs.loss
            if isinstance(model, DataParallel):
                loss = loss.mean()  # Aggregate loss across GPUs
            total_train_loss += loss.item()
            loss.backward()
            optimizer.step()
            
            train_loop.set_postfix({'batch_loss': loss.item()})
        
        avg_train_loss = total_train_loss / len(train_dataloader)
        
        # Compute validation loss
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            val_loop = tqdm(test_dataloader, desc=f"Validation Epoch {epoch + 1}")
            for batch in val_loop:
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                # Handle DataParallel loss for validation
                loss = outputs.loss
                if isinstance(model, DataParallel):
                    loss = loss.mean()  # Aggregate loss across GPUs
                total_val_loss += loss.item()
                val_loop.set_postfix({'batch_loss': loss.item()})
        
        avg_val_loss = total_val_loss / len(test_dataloader)
        
        # Store epoch results
        epoch_data.append({
            'Epoch': epoch + 1,
            'Training Loss': avg_train_loss,
            'Validation Loss': avg_val_loss
        })
        
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            # Save best model
            os.makedirs(CONFIG['output_dir'], exist_ok=True)
            if isinstance(model, DataParallel):
                model.module.save_pretrained(CONFIG['model_save_path'])
            else:
                model.save_pretrained(CONFIG['model_save_path'])
            tokenizer.save_pretrained(CONFIG['model_save_path'])
            print(f"New best model saved at epoch {epoch + 1} with validation loss: {avg_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                early_stop = True
        
        print(f"Epoch {epoch + 1}: Training Loss = {avg_train_loss:.4f}, Validation Loss = {avg_val_loss:.4f}")
    
    # Display results in a table
    results_df = pd.DataFrame(epoch_data)
    print("\nTraining Progress:")
    print(results_df.to_string(index=False))
    
    # Return the unwrapped model for evaluation
    if isinstance(model, DataParallel):
        return model.module
    return model

In [ ]:
# Main execution flow
# Initialize tokenizer
tokenizer = T5Tokenizer.from_pretrained(CONFIG['model_name'])

# Load and prepare data
dataset_path = "/kaggle/input/userstory/userstory.csv"  # Path from your input
train_dataset, test_dataset = load_and_prepare_data(dataset_path)

# Debug: Check dataset size and sample
# print(f"Train dataset size: {len(train_dataset)}")
# print(f"Test dataset size: {len(test_dataset)}")
# print("Train dataset sample:", train_dataset[0])
# print("Test dataset sample:", test_dataset[0])

# Tokenize datasets
train_dataset = tokenize_dataset(train_dataset, tokenizer)
test_dataset = tokenize_dataset(test_dataset, tokenizer)

# Debug: Check tokenized dataset format
# print("Tokenized train dataset sample:", train_dataset[0])
# print("Tokenized test dataset sample:", test_dataset[0])

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [9]:
# Train model with validation
model = train_model(train_dataset, test_dataset, tokenizer)

# Evaluate model
results = evaluate_model(model, test_dataset, tokenizer)

# Print results
print("\nEvaluation Results:")
print(f"Average BLEU Score: {results['bleu']:.4f}")
print(f"Average METEOR Score: {results['meteor']:.4f}")
print(f"Average BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"Average BERTScore Recall: {results['bertscore_recall']:.4f}")
print(f"Average BERTScore F1: {results['bertscore_f1']:.4f}")

# Print sample predictions
print("\nSample Predictions:")
for pred, ref in zip(results['predictions'][:3], results['references'][:3]):
    print(f"Predicted: {pred}")
    print(f"Reference: {ref}")
    print()

Using 2 GPUs!


Training Epoch 1:   0%|          | 0/196 [00:00<?, ?it/s]Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Validation Epoch 1: 100%|██████████| 50/50 [00:11<00:00,  4.34it/s, batch_loss=0.321]


New best model saved at epoch 1 with validation loss: 0.3537
Epoch 1: Training Loss = 4.2103, Validation Loss = 0.3537


Validation Epoch 2: 100%|██████████| 50/50 [00:11<00:00,  4.33it/s, batch_loss=0.232] 


New best model saved at epoch 2 with validation loss: 0.2675
Epoch 2: Training Loss = 0.2960, Validation Loss = 0.2675


...

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.92 seconds, 54.10 sentences/sec

Evaluation Results:
Average BLEU Score: 0.4502
Average METEOR Score: 0.6333
Average BERTScore Precision: 0.9528
Average BERTScore Recall: 0.9529
Average BERTScore F1: 0.9527

Sample Predictions:
Predicted: As a parent, I want to be able to track the location and arrival time of my child's school bus so that I can plan accordingly.
Reference: As a parent, I want to be able to view real-time updates on my child's school bus location and estimated arrival time, so that I can plan for pick-up and drop-off.

Predicted: As a high net worth individual, I want the banking system to have advanced security features, such as biometric authentication, advanced encryption methods, and personalized customer service so that I can receive personalized attention and support.
Reference: As a high net worth individual who values privacy, I want the banking system to provide top-level security features like biometric authentication, advanced encryption, strict da